# Explain runner

Update `examples/modelling/config_explain_template.json` for standard runs.

For FT -> XGB/ResNet two-step models, point `config_path` to the Step-2 config
written by `Train_FT_Embed_XGBResN.ipynb` (for example,
`config_xgb_from_ft_unsupervised.json` or `config_resn_from_ft_unsupervised_ddp.json`).
This notebook patches the config to run in explain mode and overrides the model keys.


In [ ]:
from pathlib import Path
import json

from ins_pricing.cli.utils.notebook_utils import run_from_config

# Base config to run explain on.
config_path = Path("examples/modelling/config_explain_template.json")

# For FT -> XGB/ResNet, use the Step-2 config from Train_FT_Embed_XGBResN.ipynb:
# config_path = Path("examples/modelling/config_xgb_from_ft_unsupervised.json")
# config_path = Path("examples/modelling/config_resn_from_ft_unsupervised_ddp.json")

# Models/methods to explain.
model_keys = ["xgb"]  # or ["resn"] / ["xgb", "resn"]
methods = ["permutation"]  # or ["shap"], ["integrated_gradients"], etc.

cfg = json.loads(config_path.read_text(encoding="utf-8"))

# Create a one-off explain config so training configs can run in explain mode.
runner = dict(cfg.get("runner") or {})
runner["mode"] = "explain"
cfg["runner"] = runner

explain_cfg = dict(cfg.get("explain") or {})
if model_keys:
    explain_cfg["model_keys"] = model_keys
if methods:
    explain_cfg["methods"] = methods
cfg["explain"] = explain_cfg

patched_path = config_path.with_name(f"{config_path.stem}_explain_run.json")
patched_path.write_text(json.dumps(cfg, indent=2), encoding="utf-8")

run_from_config(patched_path)
